In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import csv
from datetime import datetime, timedelta

def extract_pos_coordinates(filepath):
    """
    Extract specified columns from a position file.

    Parameters:
    - filepath: Path to the input file
    - columns_to_extract: List of column indices to extract (0-based). 
                         If None, extracts first 4 columns by default.
    - output_csv: Optional path to save results as CSV
    - column_names: Optional list of custom column names (same length as columns_to_extract)
    """
    output_csv = None
    if filepath[-1] == 'K':
        columns_to_extract = [0, 1, 2, 3]
        column_names=['GPST', 'x_y', 'y_y', 'z_y']
    else:
        columns_to_extract = [0,1,2,3]
        column_names=['GPST', 'x_y', 'y_y', 'z_y']

    extracted_rows = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip metadata and comments
            if not line or line.startswith('%'):
                continue

            parts = [p.strip() for p in line.split(',')]

            try:
                extracted_values = []
                for col_idx in columns_to_extract:
                    if col_idx < len(parts):
                        try:
                            extracted_values.append(float(parts[col_idx]))
                        except ValueError:
                            extracted_values.append(parts[col_idx])
                    else:
                        extracted_values.append(None)
                extracted_rows.append(extracted_values)

            except (IndexError, ValueError) as e:
                print(f"Warning: Skipping line due to error: {e}")
                print(f"Line content: {line}")
                continue

    # Write to CSV (with custom or default headers)
    if output_csv:
        if column_names is None:
            column_names = [f'Column_{idx}' for idx in columns_to_extract]
        elif len(column_names) != len(columns_to_extract):
            raise ValueError("column_names length must match columns_to_extract length")

        with open(output_csv, 'w', newline='') as f_out:
            writer = csv.writer(f_out)
            writer.writerow(column_names)
            writer.writerows(extracted_rows)

    return pd.DataFrame(extracted_rows, columns=column_names)

def gpst_to_datetime(week, tow):
    """
    Converts GPS Week and Time of Week (TOW) to a formatted string.
    GPS Time starts on Jan 6, 1980.
    """
    gps_epoch = datetime(1980, 1, 6, 0, 0, 0)
    
    # Calculate the exact time
    elapsed = timedelta(weeks=float(week), seconds=float(tow))
    current_time = gps_epoch + elapsed
    
    # Format to match your pos file: 'YYYY/MM/DD HH:MM:SS.ss'
    # We slice [:-4] to trim microsecond precision down to 2 decimals if needed
    return current_time.strftime('%Y/%m/%d %H:%M:%S.%f')[:-4]

def parse_rtk_stat_file(stat_file_path):
    data = []
    
    with open(stat_file_path, 'r') as f:
        current_tow = None
        current_week = None
        
        snr_values = []
        res_values = []
        
        for line in f:
            if not line.startswith("$SAT"): continue
            
            parts = line.split(',')
            
            try:
                week = int(parts[1])
                tow = float(parts[2])
            except ValueError:
                continue

            # If we moved to a new timestamp, save the previous batch
            if current_tow is not None and tow != current_tow:
                if snr_values:
                    # Convert the time for the PREVIOUS batch
                    time_str = gpst_to_datetime(current_week, current_tow)
                    
                    data.append({
                        'GPST': time_str,  # Matches your POS file column name
                        'avg_snr': np.mean(snr_values),
                        'min_snr': np.min(snr_values),
                        'max_residual': np.max(np.abs(res_values))
                    })
                snr_values = []
                res_values = []
            
            current_week = week
            current_tow = tow
            
            # --- EXTRACT FEATURES ---
            try:
                snr = float(parts[10]) # Column 10: SNR
                res = float(parts[7])  # Column 7: Residual
                
                if snr > 0:
                    snr_values.append(snr)
                    res_values.append(res)
            except (ValueError, IndexError):
                continue

    # Don't forget the very last batch
    if current_tow is not None and snr_values:
        time_str = gpst_to_datetime(current_week, current_tow)
        data.append({
            'GPST': time_str,
            'avg_snr': np.mean(snr_values),
            'min_snr': np.min(snr_values),
            'max_residual': np.max(np.abs(res_values))
        })
    

    df_snr = pd.DataFrame(data)
    df_snr['GPST'] = pd.to_datetime(df_snr['GPST']).dt.round('s')

    return df_snr

# --- HOW TO USE ---
# 1. Parse the stat file
df_snr = parse_rtk_stat_file(r'F:\zizo\RTKCorrection\src\research\data\test\SPP_BDS.stat')

# 2. Load your main POS file (assuming it has a 'GPST' or 'tow' column to match)
df_pos = extract_pos_coordinates(r'F:\zizo\RTKCorrection\src\research\data\test\kinematic_BDS')
# (You might need to calculate 'tow' from the GPST timestamp in df_pos to merge them)

print(df_snr.head())

                 GPST    avg_snr  min_snr  max_residual
0 2024-02-14 04:19:28  45.230769     41.0        2.1696
1 2024-02-14 04:19:29  45.230769     41.0        2.1409
2 2024-02-14 04:19:30  45.230769     41.0        2.2519
3 2024-02-14 04:19:31  45.153846     41.0        2.2347
4 2024-02-14 04:19:32  45.153846     41.0        2.3447


In [6]:
df_pos.head()
df_pos.to_csv('Kinematic_BDS.csv')

In [2]:
df_pos['GPST'] = pd.to_datetime(df_pos['GPST']).dt.round('s')
df = pd.merge(df_snr, df_pos, on='GPST', how='inner')
df.to_csv("SPP_BDS.csv")

In [3]:
len(df)

7962